In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

In [2]:
ds_path = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/notrees-gt.csv")
Path.is_file(ds_path)

True

In [3]:
ds = pd.read_csv(ds_path)
ds

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.187436,0.000161,0.000196,1124.690170,1020.232214,0.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
1,0.092571,0.000277,0.000303,1124.690170,1020.232214,1.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
2,0.057824,0.000337,0.000359,1124.690170,1020.232214,2.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
3,0.171885,0.000224,0.000273,1124.690170,1020.232214,3.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
4,0.110673,0.000253,0.000287,1124.690170,1020.232214,4.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
329615,0.222720,0.001223,0.001876,262.715686,200.963755,15.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
329616,0.204124,0.001236,0.001864,262.715686,200.963755,16.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
329617,0.231891,0.001070,0.001640,262.715686,200.963755,17.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
329618,0.235853,0.000894,0.001359,262.715686,200.963755,18.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26


In [4]:
ds_ar = ds.to_numpy()

In [5]:
f = np.array([1.0, 0.0, 0.0])
r = np.array([0.0, 1.0, 0.0])

In [6]:
angles = []

for row in ds_ar:
    c =  np.deg2rad(np.array([row[9], row[10], row[11]]))
    s =  np.deg2rad(np.array([row[6], row[7], row[8]]))

    # camera
    c_r_cos = np.cos(c[0])
    c_r_sin = np.sin(c[0])
    
    c_p_cos = np.cos(c[1])
    c_p_sin = np.sin(c[1])
    
    c_y_cos = np.cos(c[2])
    c_y_sin = np.sin(c[2])
    
    # sun
    s_r_cos = np.cos(s[0])
    s_r_sin = np.sin(s[0])
    
    s_p_cos = np.cos(s[1])
    s_p_sin = np.sin(s[1])
    
    s_y_cos = np.cos(s[2])
    s_y_sin = np.sin(s[2])

    #camera
    Rxc = np.array([[1.0, 0.0, 0.0], [0.0, c_r_cos, -1.0 * c_r_sin], [0.0, c_r_sin, c_r_cos]])
    Ryc = np.array([[c_p_cos, 0.0, c_p_sin], [0.0, 1.0, 0.0], [-1*c_p_sin, 0.0, c_p_cos]])
    Rzc = np.array([[c_y_cos, -1*c_y_sin, 0.0], [c_y_sin, c_y_cos, 0.0], [0.0, 0.0, 1]])

    Rc = np.matmul(Rzc, np.matmul(Ryc, Rxc))
    
    #sun
    Rxs = np.array([[1.0, 0.0, 0.0], [0.0, s_r_cos, -1.0 * s_r_sin], [0.0, s_r_sin, s_r_cos]])
    Rys = np.array([[s_p_cos, 0.0, s_p_sin], [0.0, 1.0, 0.0], [-1*s_p_sin, 0.0, s_p_cos]])
    Rzs = np.array([[s_y_cos, -1*s_y_sin, 0.0], [s_y_sin, s_y_cos, 0.0], [0.0, 0.0, 1]])

    Rs = np.matmul(Rzs, np.matmul(Rys, Rxs))

    fc = np.matmul(Rc, f)
    rc = np.matmul(Rc, r)
    fs = np.matmul(Rs, f)

    sf = np.dot(fs, fc)
    sr = np.dot(fs, rc)

    angle = np.arctan2(sr, sf)

    angles.append(angle)

In [7]:
angles_df = pd.DataFrame({"azimuth": angles})
angles_df

,azimuth
0,0.173666
1,0.173666
2,0.173666
3,0.173666
4,0.173666
...,...
329615,-1.209846
329616,-1.209846
329617,-1.209846
329618,-1.209846


In [8]:
ds_nu = ds.drop(["sunRoll", "sunPitch", "sunYaw", "camRoll", "camPitch", "camYaw"], axis=1)
ds_nu

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,grviOut
0,0.187436,0.000161,0.000196,1124.690170,1020.232214,0.0,0.20
1,0.092571,0.000277,0.000303,1124.690170,1020.232214,1.0,0.20
2,0.057824,0.000337,0.000359,1124.690170,1020.232214,2.0,0.20
3,0.171885,0.000224,0.000273,1124.690170,1020.232214,3.0,0.20
4,0.110673,0.000253,0.000287,1124.690170,1020.232214,4.0,0.20
...,...,...,...,...,...,...,...
329615,0.222720,0.001223,0.001876,262.715686,200.963755,15.0,0.26
329616,0.204124,0.001236,0.001864,262.715686,200.963755,16.0,0.26
329617,0.231891,0.001070,0.001640,262.715686,200.963755,17.0,0.26
329618,0.235853,0.000894,0.001359,262.715686,200.963755,18.0,0.26


In [9]:
out = ds["grviOut"]
ds_nu = ds_nu.drop(["grviOut"], axis=1)
ds_nu

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx
0,0.187436,0.000161,0.000196,1124.690170,1020.232214,0.0
1,0.092571,0.000277,0.000303,1124.690170,1020.232214,1.0
2,0.057824,0.000337,0.000359,1124.690170,1020.232214,2.0
3,0.171885,0.000224,0.000273,1124.690170,1020.232214,3.0
4,0.110673,0.000253,0.000287,1124.690170,1020.232214,4.0
...,...,...,...,...,...,...
329615,0.222720,0.001223,0.001876,262.715686,200.963755,15.0
329616,0.204124,0.001236,0.001864,262.715686,200.963755,16.0
329617,0.231891,0.001070,0.001640,262.715686,200.963755,17.0
329618,0.235853,0.000894,0.001359,262.715686,200.963755,18.0


In [10]:
ds_nu["azimuth"] = angles_df["azimuth"].iloc[:len(out)]
ds_nu

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,azimuth
0,0.187436,0.000161,0.000196,1124.690170,1020.232214,0.0,0.173666
1,0.092571,0.000277,0.000303,1124.690170,1020.232214,1.0,0.173666
2,0.057824,0.000337,0.000359,1124.690170,1020.232214,2.0,0.173666
3,0.171885,0.000224,0.000273,1124.690170,1020.232214,3.0,0.173666
4,0.110673,0.000253,0.000287,1124.690170,1020.232214,4.0,0.173666
...,...,...,...,...,...,...,...
329615,0.222720,0.001223,0.001876,262.715686,200.963755,15.0,-1.209846
329616,0.204124,0.001236,0.001864,262.715686,200.963755,16.0,-1.209846
329617,0.231891,0.001070,0.001640,262.715686,200.963755,17.0,-1.209846
329618,0.235853,0.000894,0.001359,262.715686,200.963755,18.0,-1.209846


In [11]:
ds_nu["grviOut"] = out.iloc[:len(out)]
ds_nu

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,azimuth,grviOut
0,0.187436,0.000161,0.000196,1124.690170,1020.232214,0.0,0.173666,0.20
1,0.092571,0.000277,0.000303,1124.690170,1020.232214,1.0,0.173666,0.20
2,0.057824,0.000337,0.000359,1124.690170,1020.232214,2.0,0.173666,0.20
3,0.171885,0.000224,0.000273,1124.690170,1020.232214,3.0,0.173666,0.20
4,0.110673,0.000253,0.000287,1124.690170,1020.232214,4.0,0.173666,0.20
...,...,...,...,...,...,...,...,...
329615,0.222720,0.001223,0.001876,262.715686,200.963755,15.0,-1.209846,0.26
329616,0.204124,0.001236,0.001864,262.715686,200.963755,16.0,-1.209846,0.26
329617,0.231891,0.001070,0.001640,262.715686,200.963755,17.0,-1.209846,0.26
329618,0.235853,0.000894,0.001359,262.715686,200.963755,18.0,-1.209846,0.26


In [12]:
ds_nu.to_csv("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/azimuth-datasets/notrees-gt-azimuth.csv", index=False)